# The files

##### There are 3 types of original files:
1. mouseNR_neuronID_neurites.xlsx -> has all the points for neurites, their branching data, and endpoints. Also has the atlas registration info for each point.

2. mouseNR_neuronID_length.xlsx -> length of axons in all areas as registered in the atlas (in this and previous files ID/id stands for the resistered area id).

3. mouseNR_neuronID.swc -> the same as mouseNR_neuronID_neurites.xlsx but without header and withouth branching and endpoint info. This file is in the folder called swc_files.

NOTE: There is one for each neuron and mouseNR_neuronID_neurites.xlsx and mouseNR_neuronID_length.xlsx are grouped by brain in one forlder per brain.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import pandas as pd
import glob
import os
from tqdm import tqdm
import re
import nrrd
from tqdm import tqdm
import shutil

midline_z = 228.5

path = str(os.getcwd())



### SWC files processing

This script processes SWC files of neurons:
1. Normalise all neurons ot the same hemisphere
2. Annotate each node with brain region information using a 3D atlas (annotation_25.nrrd and structure_tree_safe.csv).
3. Compute neurite lengths per brain region and total neuron length, saving the results in _length.xlsx and ipsi_length.xlsx.
   - The total length of the neuron is recorded only in the last row of the table.
4. Identify end-points and branch-points, update Ntype accordingly, and add a 'Structure' column.
5. Save the processed endpoints and branch points as _neurites.xlsx files.



In [2]:
id_col = "id"
acronym_col = "acronym"
full_name_col = "safe_name"         
atlas_path_col = "structure_id_path"

def read_swc(swc_path):
    """Read a raw SWC file (n, type, x, y, z, radius, parent), skipping '#' comments."""
    data = np.loadtxt(swc_path, comments="#")
    return data.reshape(1, -1) if data.ndim == 1 else data


def annotate_nodes(data, volume, structure_tree):
    """Assign a brain-region ID/Abbreviation/FullName/structure_id_path to every node."""
    nx, ny, nz = volume.shape
    sx = np.clip(np.round(data[:, 2]).astype(np.int64), 1, nx)
    sy = np.clip(np.round(data[:, 3]).astype(np.int64), 1, ny)
    sz = np.clip(np.round(data[:, 4]).astype(np.int64), 1, nz)
    region_ids = volume[sx - 1, sy - 1, sz - 1]

    df = pd.DataFrame({
        "NO": data[:, 0].astype(np.int64), 
        "Ntype": data[:, 1].astype(np.int64),
        "x": data[:, 2], "y": data[:, 3], "z": data[:, 4], 
        "Nsize": data[:, 5],
        "Nparentid": data[:, 6].astype(np.int64), 
        "id": region_ids.astype(np.int64),
    })

    lookup = structure_tree[[id_col, acronym_col, full_name_col, atlas_path_col]].rename(columns={
        id_col: "_id", acronym_col: "Abbreviation", full_name_col: "FullName", atlas_path_col: "structure_id_path",
    })
    merged = df.merge(lookup, left_on="id", right_on="_id", how="left").drop(columns=["_id"])
    for col in ("Abbreviation", "FullName", "structure_id_path"):
        merged[col] = merged[col].fillna("none")

    return merged[["NO", "Ntype", "x", "y", "z", "Nsize", "Nparentid", "id",
                    "Abbreviation", "FullName", "structure_id_path"]]




def add_endpoints_and_structure(annotated_df):
    """Flag end-points/branch-points, recode Ntype, add a 'Structure' column."""
    df = annotated_df.copy()
    node_ids, parent_ids = df["NO"].to_numpy(), df["Nparentid"].to_numpy()

    child_counts = pd.Series(parent_ids).value_counts().reindex(node_ids, fill_value=0).to_numpy() # identify number of children nodes for each parent node
    
    is_end = child_counts == 0
    is_branch = (child_counts >= 2) & (parent_ids != -1)

    orig_ntype = df["Ntype"].to_numpy()
    ntype = orig_ntype.copy()

    end_map = {2: 6, 3: 7, 4: 8}
    branch_map = {2: 5, 3: 11, 4: 12}

    for t, new_t in end_map.items():
        ntype[is_end & (orig_ntype == t)] = new_t
    for t, new_t in branch_map.items():
        ntype[is_branch & (orig_ntype == t)] = new_t

    df["Ntype"] = ntype


    structure_mapping = {
        0: "undefined", 1: "soma", 2: "axon", 3: "(basal) dendrite",
        4: "apical dendrite", 5: "axon fork point", 6: "axon end point",
        7: "(basal) dendrite end point", 8: "apical dendrite end point",
        11: "(basal) dendrite fork point", 12: "apical dendrite fork point",
    }
    df["Structure"] = df["Ntype"].map(structure_mapping).fillna("undefined")

    cols = list(df.columns)
    cols.remove("Structure")
    cols.insert(cols.index("Ntype") + 1, "Structure")
    return df[cols]


def compute_length_table(annotated_df, orig_data, resolution=25):
    """
    Per-brain-region neurite length + total neuron length.
    """
    sub = annotated_df.loc[annotated_df["Ntype"] != 1]  # exclude soma node
    parent_idx = sub["Nparentid"].to_numpy(dtype=np.int64) - 1  
    valid = parent_idx >= 0

    current_xyz = sub[["x", "y", "z"]].to_numpy()[valid] * resolution
    parent_xyz = orig_data[:, 2:5][parent_idx[valid]] * resolution
    distances = np.linalg.norm(current_xyz - parent_xyz, axis=1)
    total_length = float(distances.sum())

    lengths_df = pd.DataFrame({
        "ID": sub["id"].to_numpy()[valid],
        "Length": distances,
        "Abbreviation": sub["Abbreviation"].to_numpy()[valid],
        "FullName": sub["FullName"].to_numpy()[valid],
    })
    result = lengths_df.groupby("ID", sort=False).agg(
        Length=("Length", "sum"),
        Abbreviation=("Abbreviation", "first"),
        FullName=("FullName", "first"),
    ).reset_index()

    result["ID"] = result["ID"].astype(str)
    result["totalLength_um"] = np.nan
    result.loc[len(result)] = ["", np.nan, "", "", total_length]
    return result


In [4]:

structure_tree = pd.read_csv(f"{path}/structure_tree_safe.csv")
volume, _ = nrrd.read(f"{path}/annotation_25.nrrd")

swc_files = sorted(glob.glob(f"{path}/raw_snr_data/*.swc"))

for swc_path in tqdm(swc_files, desc="Processing neurons"):
    file_name = os.path.basename(swc_path)
    full_neuron_id = os.path.splitext(file_name)[0]
    mouse_id = full_neuron_id.split("_")[0]

    preprocessed_dir = f"{path}/preprocessed_data/{mouse_id}"
    os.makedirs(preprocessed_dir, exist_ok=True)


    original_data = read_swc(swc_path)
    soma_z = original_data[0, 4]

    whole_annotated = annotate_nodes(original_data, volume, structure_tree)

    # end-points / branch-points
    neurites = add_endpoints_and_structure(whole_annotated)

    # whole-neuron length table
    compute_length_table(whole_annotated, original_data).to_excel(
        os.path.join(preprocessed_dir, f"{full_neuron_id}_length.xlsx"), index=False)

    # only build the ipsi-only table if an axon endpoint actually crosses the
    # midline (padded by one voxel to absorb boundary jitter)
    axon_endpoint_z = neurites.loc[neurites["Ntype"] == 6, "z"]
    if soma_z < midline_z:
        has_contra = (axon_endpoint_z > midline_z + 25).any()
        ipsi_idx = original_data[:, 4] <= midline_z
    else:
        has_contra = (axon_endpoint_z < midline_z - 25).any()
        ipsi_idx = original_data[:, 4] >= midline_z

    if has_contra:
        ipsi_data = original_data[ipsi_idx]
        ipsi_annotated = annotate_nodes(ipsi_data, volume, structure_tree)
        compute_length_table(ipsi_annotated, original_data).to_excel(
            os.path.join(preprocessed_dir, f"{full_neuron_id}_ipsi_length.xlsx"), index=False)

    if soma_z < midline_z:
        neurites['z'] = midline_z * 2 - neurites['z']

    neurites.to_excel(os.path.join(preprocessed_dir, f"{full_neuron_id}_neurites.xlsx"), index=False)



print("All neurons processed.")

Processing neurons: 100%|██████████| 237/237 [23:58<00:00,  6.07s/it]

All neurons processed.


### Master list of all endpoint counts and axon length

In [5]:

# load the structure file
structure_tree = pd.read_csv(f"{path}/structure_tree_safe.csv")


# areas of interest
areas = [
    'ILA','PL', 'ORB', 'AUD', 'TEa', 'ECT', 'PERI', 'VISC', 
    'AI', 'MO', 'SS', 'ACA', 'PTLp','VIS',  'RSP', 'GU', 'VISpor',
    'OLF', 'CTXsp', 
        'CLA', 'EP', 
        #AMY,
        'BLA', 'LA', 'BMA', 'PA', 
    #'STR', 
        'STRv', 'STRd', 
    'PAL', 
    #'HPF', 
        'HIP', 'RHP',
            'SUB', 'PAR', 'POST', 'PRE', 'ENT' #RHP is to broad with ENT inclusion maybe
]


area_id_paths = {area: structure_tree.loc[structure_tree[acronym_col] == area, "structure_id_path"].values[0] 
         for area in areas}

# for length only the highest hierarhies should be used and lower ones if available should be summed to that
lists_of_children = {
    area: structure_tree.loc[
        structure_tree[atlas_path_col].str.startswith(area_id_paths[area]), "id"
    ].tolist()
    for area in areas
}


In [6]:

list_of_brain_folders = glob.glob(str(path) + '/preprocessed_data/*/') # all the forlders are the brain ID

# create lists for all rows of the master_list_endpoints and master_list_lengths dataset to reduce computation
all_rows_endpoints = []
all_rows_lengths = []

for folder in tqdm(list_of_brain_folders, desc="Processing Files"):
    neuron_files = glob.glob(folder + "*") # Get all .xlsx files in the folder
    for file in neuron_files:
        if "_neurites.xlsx" in file:
            ID = os.path.basename(file).replace("_neurites.xlsx","") # get neuron ID expressed as mouseID_neuronID
            neuronNR = ID.split("_")[1]
            mouseID = ID.split("_")[0]


            endpoint_info = pd.read_excel(file, usecols = ['Ntype', 'x', 'y', 'z',  'Nparentid','id', 'Abbreviation', 'structure_id_path']) # read the neuron file
            endpoint_info_both_hem = endpoint_info[endpoint_info["Ntype"] == 6] # filter the file for only axon endpoints represented by Ntype 6
            # only use endpoints from the soma hemisphere
            endpoint_info = endpoint_info_both_hem[endpoint_info_both_hem["z"] > midline_z]
            #endpoint_info['structure_id_path'] = endpoint_info['structure_id_path'].apply(format_id_path)

            # now need to use structure id paths: if id path contains the path part of interest to be assigned to an area then we add that to an area
            target_endpoint_list = [] # cerate an empty list that will have info for each area in the previously created empty dataframe
            paths_in_file = endpoint_info['structure_id_path'].tolist()
            for area in areas:
                target_path = area_id_paths[area]
                # counts how many paths have target area path inside them
                count = sum(1 for p in paths_in_file if target_path in str(p))
                target_endpoint_list.append(count)


            all_rows_endpoints.append([mouseID, neuronNR, ID, *target_endpoint_list]) # create a list of lists for each neuron

            # find the length file for this neuron
            # identify if the inpsi file exists for it and if it does proceed with that file
            # if not find another length file and proceed with that
            ### The point is that we only care about projections to the ipsilateral hemisphere which is already the case for neurons that dont even project contralaterally but is a specific case for those that do
            length_file = glob.glob(folder + ID +"_" +"*length.xlsx")
            for i in length_file:
                if "ipsi" in i:
                    length_file_to_pick = i
                    break
                elif "ipsi" not in i:
                    length_file_to_pick = i


            
            length_info = pd.read_excel(length_file_to_pick, usecols=['ID'	,'Length'])
            target_length_list = []
            for area in areas:
                target_ids = lists_of_children[area]
                # sum lengths if 'ID' column has any of the ids in the list
                length_info_for_area = length_info[length_info['ID'].isin(target_ids)]
                length = length_info_for_area['Length'].sum()
                target_length_list.append(length)
       
            
            all_rows_lengths.append([mouseID, neuronNR, ID, *target_length_list]) # create a list of lists for each neuron



# the masterlist with the previously outlined columns and rows stored in lists
master_list_endpoints_new = pd.DataFrame(all_rows_endpoints, columns=['mouseID', 'neuronNR', 'ID']+[a + '_endpoint' for a in areas])
master_list_length_new = pd.DataFrame(all_rows_lengths, columns=['mouseID', 'neuronNR', 'ID']+[a + '_length' for a in areas])


Processing Files: 100%|██████████| 11/11 [10:52<00:00, 59.28s/it]


In [7]:
# merging files
# merging of length and endpoint data into a master file
merged_df = pd.merge(master_list_endpoints_new, master_list_length_new, on=["mouseID", "neuronNR", "ID"])


cols = merged_df.columns
id_cols = ["mouseID", "neuronNR", "ID"]
endpoint_cols = [col for col in cols if col.endswith("_endpoint")]
length_cols = [col for col in cols if col.endswith("_length")]

# Interleave columns
interleaved_cols = id_cols + [c for pair in zip(endpoint_cols, length_cols) for c in pair]
merged_df = merged_df[interleaved_cols]




master_list = merged_df.rename(columns = {"ID": "neuron_ID"})

master_list["PFC_length"] = master_list["PL_length"] + master_list["ILA_length"]
master_list["PFC_endpoint"] = master_list["PL_endpoint"] + master_list["ILA_endpoint"]

master_list["AMY_length"] = master_list["BLA_length"] + master_list["LA_length"] +master_list["BMA_length"] + master_list["PA_length"]
master_list["AMY_endpoint"] = master_list["BLA_endpoint"] + master_list["LA_endpoint"] +master_list["BMA_endpoint"] + master_list["PA_endpoint"]


master_list["RHP(no_EC)_length"] = master_list["RHP_length"] - master_list["ENT_length"] 
master_list["RHP(no_EC)_endpoint"] = master_list["RHP_endpoint"] - master_list["ENT_endpoint"] 

master_list["VIS_length"] = master_list["VIS_length"] - master_list["VISpor_length"]
master_list["VIS_endpoint"] = master_list["VIS_endpoint"] - master_list["VISpor_endpoint"]

master_list

,mouseID,neuronNR,neuron_ID,ILA_endpoint,ILA_length,PL_endpoint,PL_length,ORB_endpoint,ORB_length,AUD_endpoint,...,PRE_endpoint,PRE_length,ENT_endpoint,ENT_length,PFC_length,PFC_endpoint,AMY_length,AMY_endpoint,RHP(no_EC)_length,RHP(no_EC)_endpoint
0,1056,2,1056_2,1,934.002583,4,3122.303074,20,10442.910211,1,...,0,0.000000,84,40171.050018,4056.305657,5,32023.655292,34,23532.363782,41
1,1056,41,1056_41,6,922.495470,5,2276.703226,29,16180.451953,0,...,0,0.000000,17,17812.344630,3199.198697,11,20969.945068,31,9746.391537,21
2,1056,6,1056_6,0,1045.593339,6,2406.538107,12,12320.574242,8,...,0,0.000000,114,53396.803659,3452.131446,6,9255.326473,5,743.547347,2
3,1056,26,1056_26,0,0.000000,0,0.000000,0,0.000000,1,...,0,0.000000,67,31366.874409,0.000000,0,4410.365233,11,2673.955984,7
4,1056,31,1056_31,20,12749.912449,8,1930.642441,28,18397.953216,0,...,0,0.000000,86,37993.727864,14680.554890,28,62458.922187,83,6109.023824,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
232,1849931,6,1849931_6,0,0.000000,0,0.000000,0,0.000000,31,...,0,1120.317152,19,18147.078778,0.000000,0,0.000000,0,15430.791130,21
233,1849931,10,1849931_10,4,2101.035393,10,5141.513257,2,1060.607202,7,...,0,237.208737,16,24259.827278,7242.548650,14,2515.250924,6,8953.778731,5
234,1849931,14,1849931_14,0,1107.388103,6,2742.674196,3,3112.813346,25,...,0,106.906774,23,27778.764731,3850.062299,6,16519.039178,21,3609.052154,3
235,1849931,13,1849931_13,0,1524.024277,2,396.262159,19,13435.673226,0,...,1,282.992243,23,26599.567383,1920.286436,2,1389.298581,2,10306.470144,17


In [8]:


cols = merged_df.columns
id_cols = ["mouseID", "neuronNR", "ID"]
endpoint_cols = [col for col in cols if col.endswith("_endpoint")]
length_cols = [col for col in cols if col.endswith("_length")]

# Interleave columns
interleaved_cols = id_cols + [c for pair in zip(endpoint_cols, length_cols) for c in pair]
merged_df = merged_df[interleaved_cols]




master_list = merged_df.rename(columns = {"ID": "neuron_ID"})

master_list["PFC_length"] = master_list["PL_length"] + master_list["ILA_length"]
master_list["PFC_endpoint"] = master_list["PL_endpoint"] + master_list["ILA_endpoint"]

master_list["AMY_length"] = master_list["BLA_length"] + master_list["LA_length"] +master_list["BMA_length"] + master_list["PA_length"]
master_list["AMY_endpoint"] = master_list["BLA_endpoint"] + master_list["LA_endpoint"] +master_list["BMA_endpoint"] + master_list["PA_endpoint"]


master_list["RHP(no_EC)_length"] = master_list["RHP_length"] - master_list["ENT_length"] 
master_list["RHP(no_EC)_endpoint"] = master_list["RHP_endpoint"] - master_list["ENT_endpoint"] 

master_list["VIS_length"] = master_list["VIS_length"] - master_list["VISpor_length"]
master_list["VIS_endpoint"] = master_list["VIS_endpoint"] - master_list["VISpor_endpoint"]

master_list

,mouseID,neuronNR,neuron_ID,ILA_endpoint,ILA_length,PL_endpoint,PL_length,ORB_endpoint,ORB_length,AUD_endpoint,...,PRE_endpoint,PRE_length,ENT_endpoint,ENT_length,PFC_length,PFC_endpoint,AMY_length,AMY_endpoint,RHP(no_EC)_length,RHP(no_EC)_endpoint
0,1056,2,1056_2,1,934.002583,4,3122.303074,20,10442.910211,1,...,0,0.000000,84,40171.050018,4056.305657,5,32023.655292,34,23532.363782,41
1,1056,41,1056_41,6,922.495470,5,2276.703226,29,16180.451953,0,...,0,0.000000,17,17812.344630,3199.198697,11,20969.945068,31,9746.391537,21
2,1056,6,1056_6,0,1045.593339,6,2406.538107,12,12320.574242,8,...,0,0.000000,114,53396.803659,3452.131446,6,9255.326473,5,743.547347,2
3,1056,26,1056_26,0,0.000000,0,0.000000,0,0.000000,1,...,0,0.000000,67,31366.874409,0.000000,0,4410.365233,11,2673.955984,7
4,1056,31,1056_31,20,12749.912449,8,1930.642441,28,18397.953216,0,...,0,0.000000,86,37993.727864,14680.554890,28,62458.922187,83,6109.023824,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
232,1849931,6,1849931_6,0,0.000000,0,0.000000,0,0.000000,31,...,0,1120.317152,19,18147.078778,0.000000,0,0.000000,0,15430.791130,21
233,1849931,10,1849931_10,4,2101.035393,10,5141.513257,2,1060.607202,7,...,0,237.208737,16,24259.827278,7242.548650,14,2515.250924,6,8953.778731,5
234,1849931,14,1849931_14,0,1107.388103,6,2742.674196,3,3112.813346,25,...,0,106.906774,23,27778.764731,3850.062299,6,16519.039178,21,3609.052154,3
235,1849931,13,1849931_13,0,1524.024277,2,396.262159,19,13435.673226,0,...,1,282.992243,23,26599.567383,1920.286436,2,1389.298581,2,10306.470144,17


# All soma coordinates

In [9]:
path = str(os.getcwd())

list_of_files = glob.glob(str(path)+'/preprocessed_data/*/*neurites.xlsx') 

all_rows = []
for file_name in tqdm(list_of_files):
    soma_row = pd.read_excel(file_name, usecols=["Ntype", "x", "y", "z"], nrows=1)

    if soma_row.loc[0, "Ntype"] != 1:
        # if soma wasn't the first row for this file fall back to a full scan
        print(f"Warning: soma not in row 0 for {file_name}, using fallback")
        neuron_data = pd.read_excel(file_name, usecols=["Ntype", "x", "y", "z"])
        soma_row = neuron_data[neuron_data["Ntype"] == 1]

    x_soma, y_soma, z_soma = soma_row.iloc[0][["x", "y", "z"]] * 25 # scale to 25um
    ID = file_name.split("/")[-1].replace("_neurites.xlsx","")

    soma = [ID,  x_soma,  y_soma,  z_soma]
    all_rows.append(soma)

soma_df = pd.DataFrame(all_rows,columns=["neuron_ID","x", "y", "z"]) # new dataset for soma coordinates


100%|██████████| 237/237 [00:00<00:00, 314.34it/s]


In [10]:

joint_master_soma = pd.merge(master_list, soma_df, how="left", on="neuron_ID")
joint_master_soma



,mouseID,neuronNR,neuron_ID,ILA_endpoint,ILA_length,PL_endpoint,PL_length,ORB_endpoint,ORB_length,AUD_endpoint,...,ENT_length,PFC_length,PFC_endpoint,AMY_length,AMY_endpoint,RHP(no_EC)_length,RHP(no_EC)_endpoint,x,y,z
0,1056,2,1056_2,1,934.002583,4,3122.303074,20,10442.910211,1,...,40171.050018,4056.305657,5,32023.655292,34,23532.363782,41,9662.400000,3506.800000,9755.300000
1,1056,41,1056_41,6,922.495470,5,2276.703226,29,16180.451953,0,...,17812.344630,3199.198697,11,20969.945068,31,9746.391537,21,9791.900000,3825.000000,9756.800000
2,1056,6,1056_6,0,1045.593339,6,2406.538107,12,12320.574242,8,...,53396.803659,3452.131446,6,9255.326473,5,743.547347,2,9680.900000,3897.000000,9768.100000
3,1056,26,1056_26,0,0.000000,0,0.000000,0,0.000000,1,...,31366.874409,0.000000,0,4410.365233,11,2673.955984,7,9844.700000,3033.500000,9482.400000
4,1056,31,1056_31,20,12749.912449,8,1930.642441,28,18397.953216,0,...,37993.727864,14680.554890,28,62458.922187,83,6109.023824,9,9232.800000,4481.300000,10019.400000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
232,1849931,6,1849931_6,0,0.000000,0,0.000000,0,0.000000,31,...,18147.078778,0.000000,0,0.000000,0,15430.791130,21,9884.758759,3947.212982,9386.614227
233,1849931,10,1849931_10,4,2101.035393,10,5141.513257,2,1060.607202,7,...,24259.827278,7242.548650,14,2515.250924,6,8953.778731,5,9827.541351,4194.146347,9382.127380
234,1849931,14,1849931_14,0,1107.388103,6,2742.674196,3,3112.813346,25,...,27778.764731,3850.062299,6,16519.039178,21,3609.052154,3,9761.907959,4358.236694,9349.091339
235,1849931,13,1849931_13,0,1524.024277,2,396.262159,19,13435.673226,0,...,26599.567383,1920.286436,2,1389.298581,2,10306.470144,17,9809.710693,4318.677521,9393.083954


In [11]:
joint_master_soma.loc[(joint_master_soma["ORB_endpoint"] > 0) & (joint_master_soma["RSP_endpoint"] > 0), "proj"] = "rsp_orb"
joint_master_soma.loc[(joint_master_soma["ORB_endpoint"] > 0) & (joint_master_soma["RSP_endpoint"] == 0), "proj"] = "orb"
joint_master_soma.loc[(joint_master_soma["ORB_endpoint"] == 0) & (joint_master_soma["RSP_endpoint"] > 0), "proj"] = "rsp"
joint_master_soma

,mouseID,neuronNR,neuron_ID,ILA_endpoint,ILA_length,PL_endpoint,PL_length,ORB_endpoint,ORB_length,AUD_endpoint,...,PFC_length,PFC_endpoint,AMY_length,AMY_endpoint,RHP(no_EC)_length,RHP(no_EC)_endpoint,x,y,z,proj
0,1056,2,1056_2,1,934.002583,4,3122.303074,20,10442.910211,1,...,4056.305657,5,32023.655292,34,23532.363782,41,9662.400000,3506.800000,9755.300000,rsp_orb
1,1056,41,1056_41,6,922.495470,5,2276.703226,29,16180.451953,0,...,3199.198697,11,20969.945068,31,9746.391537,21,9791.900000,3825.000000,9756.800000,rsp_orb
2,1056,6,1056_6,0,1045.593339,6,2406.538107,12,12320.574242,8,...,3452.131446,6,9255.326473,5,743.547347,2,9680.900000,3897.000000,9768.100000,rsp_orb
3,1056,26,1056_26,0,0.000000,0,0.000000,0,0.000000,1,...,0.000000,0,4410.365233,11,2673.955984,7,9844.700000,3033.500000,9482.400000,rsp
4,1056,31,1056_31,20,12749.912449,8,1930.642441,28,18397.953216,0,...,14680.554890,28,62458.922187,83,6109.023824,9,9232.800000,4481.300000,10019.400000,orb
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
232,1849931,6,1849931_6,0,0.000000,0,0.000000,0,0.000000,31,...,0.000000,0,0.000000,0,15430.791130,21,9884.758759,3947.212982,9386.614227,rsp
233,1849931,10,1849931_10,4,2101.035393,10,5141.513257,2,1060.607202,7,...,7242.548650,14,2515.250924,6,8953.778731,5,9827.541351,4194.146347,9382.127380,rsp_orb
234,1849931,14,1849931_14,0,1107.388103,6,2742.674196,3,3112.813346,25,...,3850.062299,6,16519.039178,21,3609.052154,3,9761.907959,4358.236694,9349.091339,rsp_orb
235,1849931,13,1849931_13,0,1524.024277,2,396.262159,19,13435.673226,0,...,1920.286436,2,1389.298581,2,10306.470144,17,9809.710693,4318.677521,9393.083954,rsp_orb


In [12]:
merged_df = joint_master_soma.copy()

merged_df["comment"] = "NA"

merged_df

,mouseID,neuronNR,neuron_ID,ILA_endpoint,ILA_length,PL_endpoint,PL_length,ORB_endpoint,ORB_length,AUD_endpoint,...,PFC_endpoint,AMY_length,AMY_endpoint,RHP(no_EC)_length,RHP(no_EC)_endpoint,x,y,z,proj,comment
0,1056,2,1056_2,1,934.002583,4,3122.303074,20,10442.910211,1,...,5,32023.655292,34,23532.363782,41,9662.400000,3506.800000,9755.300000,rsp_orb,NA
1,1056,41,1056_41,6,922.495470,5,2276.703226,29,16180.451953,0,...,11,20969.945068,31,9746.391537,21,9791.900000,3825.000000,9756.800000,rsp_orb,NA
2,1056,6,1056_6,0,1045.593339,6,2406.538107,12,12320.574242,8,...,6,9255.326473,5,743.547347,2,9680.900000,3897.000000,9768.100000,rsp_orb,NA
3,1056,26,1056_26,0,0.000000,0,0.000000,0,0.000000,1,...,0,4410.365233,11,2673.955984,7,9844.700000,3033.500000,9482.400000,rsp,NA
4,1056,31,1056_31,20,12749.912449,8,1930.642441,28,18397.953216,0,...,28,62458.922187,83,6109.023824,9,9232.800000,4481.300000,10019.400000,orb,NA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
232,1849931,6,1849931_6,0,0.000000,0,0.000000,0,0.000000,31,...,0,0.000000,0,15430.791130,21,9884.758759,3947.212982,9386.614227,rsp,NA
233,1849931,10,1849931_10,4,2101.035393,10,5141.513257,2,1060.607202,7,...,14,2515.250924,6,8953.778731,5,9827.541351,4194.146347,9382.127380,rsp_orb,NA
234,1849931,14,1849931_14,0,1107.388103,6,2742.674196,3,3112.813346,25,...,6,16519.039178,21,3609.052154,3,9761.907959,4358.236694,9349.091339,rsp_orb,NA
235,1849931,13,1849931_13,0,1524.024277,2,396.262159,19,13435.673226,0,...,2,1389.298581,2,10306.470144,17,9809.710693,4318.677521,9393.083954,rsp_orb,NA


In [13]:



# add injection place column
orb_inj = "1056|1057|1807141|1814612"
rsp_inj = "1135|1849931|1849929|1170|1171|1911916|1114|1137"

# add injection side based on the neuron ID column
merged_df.loc[merged_df["neuron_ID"].str.contains(orb_inj), "injection"] = "ORB"
merged_df.loc[merged_df["neuron_ID"].str.contains(rsp_inj), "injection"] = "RSP"

# add a comment if injection site doesn't meet the filter - NOT USED ANYMORE
# merged_df.loc[(merged_df["injection"] == "RSP") & (merged_df["RSP_endpoint"] < 2) & (merged_df["RSP_length"] < 3000), 'comment'] = "injection area doesn't meet filter requirement"
# merged_df.loc[(merged_df["injection"] == "ORB") & (merged_df["ORB_endpoint"] < 2) & (merged_df["ORB_length"] < 3000), 'comment'] = "injection area doesn't meet filter requirement"

# if there are 0 endpoints in the injection area
merged_df.loc[(merged_df["injection"] == "RSP") & (merged_df["RSP_endpoint"] == 0), 'comment'] = merged_df.loc[(merged_df["injection"] == "RSP") & (merged_df["RSP_endpoint"] == 0), 'comment'] + " injection area has 0 endpoints"
merged_df.loc[(merged_df["injection"] == "ORB") & (merged_df["ORB_endpoint"] == 0), 'comment'] = merged_df.loc[(merged_df["injection"] == "ORB") & (merged_df["ORB_endpoint"] == 0), 'comment'] + " injection area has 0 endpoints"

# if there are 0 length in the injection area
merged_df.loc[(merged_df["injection"] == "RSP") & (merged_df["RSP_length"] == 0), 'comment'] = merged_df.loc[(merged_df["injection"] == "RSP") & (merged_df["RSP_length"] == 0), 'comment'] + " injection area has 0 length"
merged_df.loc[(merged_df["injection"] == "ORB") & (merged_df["ORB_length"] == 0), 'comment'] = merged_df.loc[(merged_df["injection"] == "ORB") & (merged_df["ORB_length"] == 0), 'comment'] + " injection area has 0 length"


# neurons outside EC 1807141_32, 1135_17, 1056_22; 1807141_24 is possibly layer 3 neuron due to its possition and projections to hippocampus
outside_ec = ["1056_22", "1135_17", "1807141_32", "1114_76", "1114_57", "1114_77","1114_93", "1114_60",  "1114_55", "1114_53", "1807141_24"]
merged_df.loc[merged_df["neuron_ID"].isin(outside_ec), 'comment'] = "outside of ECL5a"


# bad (low confidence) tracing
incomplete = ids = [
    "1807141_36",
    "1807141_22",
    "1135_14",
    "1057_11",
    "1057_10",
    "1056_23",
    "1056_33",
    "1056_39",
    "1056_36",

    "1057_5",
    "1056_40",
    "1056_14",
    "1135_15",
    "1135_13",
    "1135_8",
    "1057_18",
    "1057_15",
    "1849929_4",
    "1849929_38",

    "1114_10",
    "1114_23",
    "1114_62",
    "1114_74",
    "1114_85",
    "1171_2",
    "1171_104",


    
    "1137_19",
    "1137_24",
    "1137_33",
    "1137_44",
    "1137_57",
    "1137_64",
    "1137_63",
    "1137_75",
    "1137_66",
    "1137_65",
    "1137_86",
    "1137_88",
    "1137_22",
    "1137_41",
    "1137_78",
    "1137_55",
    "1137_51",
    "1137_25",
    "1137_21",

    "1849931_7"
]



merged_df.loc[merged_df["neuron_ID"].isin(incomplete), 'comment'] = "bad tracing"






merged_df.loc[
    (merged_df["comment"]!="NA"),
    "proj"
] = ""


merged_df = merged_df.loc[(merged_df["proj"].isin(["orb","rsp_orb","rsp"])) | (merged_df["neuron_ID"].isin(["1114_10","1849931_7"]))]
merged_df.to_csv(path + "/preprocessed_data/master_detailed_comment.csv", index = False)
print("Data saved as master_detailed_comment.csv")
merged_df

Data saved as master_detailed_comment.csv


,mouseID,neuronNR,neuron_ID,ILA_endpoint,ILA_length,PL_endpoint,PL_length,ORB_endpoint,ORB_length,AUD_endpoint,...,AMY_length,AMY_endpoint,RHP(no_EC)_length,RHP(no_EC)_endpoint,x,y,z,proj,comment,injection
0,1056,2,1056_2,1,934.002583,4,3122.303074,20,10442.910211,1,...,32023.655292,34,23532.363782,41,9662.400000,3506.800000,9755.300000,rsp_orb,NA,ORB
1,1056,41,1056_41,6,922.495470,5,2276.703226,29,16180.451953,0,...,20969.945068,31,9746.391537,21,9791.900000,3825.000000,9756.800000,rsp_orb,NA,ORB
2,1056,6,1056_6,0,1045.593339,6,2406.538107,12,12320.574242,8,...,9255.326473,5,743.547347,2,9680.900000,3897.000000,9768.100000,rsp_orb,NA,ORB
4,1056,31,1056_31,20,12749.912449,8,1930.642441,28,18397.953216,0,...,62458.922187,83,6109.023824,9,9232.800000,4481.300000,10019.400000,orb,NA,ORB
5,1056,35,1056_35,19,7366.175782,6,2113.191433,17,11338.695156,0,...,21264.042304,25,1668.630472,7,9695.200000,3943.600000,9756.100000,orb,NA,ORB
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
232,1849931,6,1849931_6,0,0.000000,0,0.000000,0,0.000000,31,...,0.000000,0,15430.791130,21,9884.758759,3947.212982,9386.614227,rsp,NA,RSP
233,1849931,10,1849931_10,4,2101.035393,10,5141.513257,2,1060.607202,7,...,2515.250924,6,8953.778731,5,9827.541351,4194.146347,9382.127380,rsp_orb,NA,RSP
234,1849931,14,1849931_14,0,1107.388103,6,2742.674196,3,3112.813346,25,...,16519.039178,21,3609.052154,3,9761.907959,4358.236694,9349.091339,rsp_orb,NA,RSP
235,1849931,13,1849931_13,0,1524.024277,2,396.262159,19,13435.673226,0,...,1389.298581,2,10306.470144,17,9809.710693,4318.677521,9393.083954,rsp_orb,NA,RSP
